In [1]:
import numpy as np
import pandas as pd
import torch
import faiss
import time

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("PyTorch:", torch.__version__)
print("GPU available:", torch.backends.mps.is_available())
print("FAISS:", faiss.__version__)

PyTorch: 2.15.0
GPU available: True
FAISS: 1.15.1


In [2]:
EMBEDDING_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL)

print("Embedding Model Dimension -> ",embedder.get_sentence_embedding_dimension())

Embedding Model Dimension ->  384


/var/folders/lv/p9vlph914d5554g_z5tcrsdm0000gn/T/ipykernel_16278/1484055190.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding Model Dimension -> ",embedder.get_sentence_embedding_dimension())


In [3]:
sentences = ["This is an example sentence", "Each sentence is converted", 'Hello World this is an embedding example!']

embeddings = embedder.encode(
    sentences,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Number of Sentences -> ", len(sentences))
print("Embeddings Shape -> ", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of Sentences ->  3
Embeddings Shape ->  torch.Size([3, 384])


In [4]:
embeddings

tensor([[ 0.0677,  0.0635,  0.0487,  ...,  0.0381,  0.0600, -0.0422],
        [ 0.0864,  0.1028,  0.0054,  ...,  0.0703,  0.0749, -0.0842],
        [ 0.0034, -0.0100, -0.0022,  ...,  0.0666,  0.0543,  0.0588]],
       device='mps:0')

In [5]:
# Sentence A: I love playing cricket.
# Sentence B: Cricket is my favourite sport.
# Sentence C: The weather is very hot today.
# Compute Euclidian Distance, Dot Product and Cosine Similarity

sentences = ["I love playing cricket", "Cricket is my favourite sport", "The weather is very hot today"]

embeddings = embedder.encode(
    sentences,
    convert_to_numpy=True,
    show_progress_bar=True
)


dot_product = np.dot(embeddings[2], embeddings[1])
euclidian_distance = np.linalg.norm(embeddings[2] - embeddings[1])
cosine_similarity = np.dot(embeddings[2], embeddings[1]) / (np.linalg.norm(embeddings[2]) * np.linalg.norm(embeddings[1]))

print("Dot Product -> ", dot_product)
print("Euclidian Distance -> ", euclidian_distance)
print("Cosine Similarity -> ", cosine_similarity)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Dot Product ->  0.12007312
Euclidian Distance ->  1.3265948
Cosine Similarity ->  0.12007312


In [6]:
documents = [
    "The university library closes at 10 PM.",
    "Students must maintain 75 percent attendance.",
    "The examination registration deadline is November 15.",
    "The hostel provides vegetarian and non-vegetarian meals.",
    "Scholarships are available for eligible students.",
    "The cricket team won the inter-college tournament.",
    "The computer science department offers AI courses.",
    "The university provides free Wi-Fi in the library.",
    "Students can borrow five books at a time.",
    "Hostel residents must return before 11 PM."
]

documnet_embeddings = embedder.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embeddings Shape -> ", documnet_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Shape ->  (10, 384)


In [7]:
def cosine_similarity(x,y):

  numerator = np.dot(x,y)
  denominator = np.linalg.norm(x) * np.linalg.norm(y)

  if denominator == 0:
    return 0.0

  return numerator / denominator

sim_1_2 = cosine_similarity(documnet_embeddings[0], documnet_embeddings[1])

sim_1_3 = cosine_similarity(documnet_embeddings[0], documnet_embeddings[2])

print("Similarity between Document 1 and Document 2 -> ", sim_1_2)
print("Similarity between Document 1 and Document 3 -> ", sim_1_3)


Similarity between Document 1 and Document 2 ->  0.15265742
Similarity between Document 1 and Document 3 ->  0.32226226


In [7]:
def semantic_serch(query, documents, embeddings, top_k=3):
  query_embedding = embedder.encode(query)

  similarities = np.array([cosine_similarity(query_embedding, doc_embedding) for doc_embedding in embeddings])

  top_indices = np.argsort(similarities)[::-1][:top_k]

  result = []

  for i in top_indices:
    result.append({
        "document": documents[i],
        "similarity": similarities[i]
    })

  return result

In [8]:
queries = [
    "When does the university library close?",
    "What is the attendance requirement?",
    "What is the deadline for registration?",
]

for q in queries:
  print("Query -> ", q)

  results = semantic_serch(q, documents, documnet_embeddings)

  for r in results:
    print("Document -> ", r["document"])
    print("Similarity -> ", r["similarity"])
    print()

  print("******************************************")

Query ->  When does the university library close?
Document ->  The university library closes at 10 PM.
Similarity ->  0.8670937

Document ->  The university provides free Wi-Fi in the library.
Similarity ->  0.42155945

Document ->  Hostel residents must return before 11 PM.
Similarity ->  0.36943185

******************************************
Query ->  What is the attendance requirement?
Document ->  Students must maintain 75 percent attendance.
Similarity ->  0.6330825

Document ->  Hostel residents must return before 11 PM.
Similarity ->  0.38908535

Document ->  The examination registration deadline is November 15.
Similarity ->  0.37990284

******************************************
Query ->  What is the deadline for registration?
Document ->  The examination registration deadline is November 15.
Similarity ->  0.7124547

Document ->  Hostel residents must return before 11 PM.
Similarity ->  0.27636024

Document ->  The university library closes at 10 PM.
Similarity ->  0.22424908

In [18]:
with open("book.txt", "r", encoding="utf-8") as file:
    lines_list = [line.strip() for line in file if line.strip()]
    print(len(lines_list))

10949


In [20]:
for i in range(5):
    print(i,lines_list[i])

0 The Project Gutenberg eBook of Paradise Lost
1 This eBook is for the use of anyone anywhere in the United States and
2 most other parts of the world at no cost and with almost no restrictions
3 whatsoever. You may copy it, give it away or re-use it under the terms
4 of the Project Gutenberg License included with this eBook or online


### ------------------------------------------------------------------------------------------------------------------------------------------

In [21]:
sample_text = """

The university library opens at 8 AM and closes at 10 PM.

Students can borrow five books at a time.

The borrowing period is fourteen days.

Late returns attract a fine of five rupees per day.

Students must maintain seventy-five percent attendance.

The examination registration deadline is November 15.

Scholarships are available for eligible students.

"""




In [22]:
def fixed_size_chunking(text,chunk_size = 100):

  chunks = []

  for start in range(0, len(text), chunk_size):
    chunk = text[start:start+chunk_size]
    chunks.append(chunk)

  return chunks

chunks = fixed_size_chunking(sample_text)

print("Number of Chunks -> ", len(chunks))
print("First Chunk -> ", chunks[0])

Number of Chunks ->  4
First Chunk ->  

The university library opens at 8 AM and closes at 10 PM.

Students can borrow five books at a tim


In [23]:
def overlapping_chunks(text,chunk_size=100,overlap_size = 20):

  chunks = []

  for start in range(0, len(text), chunk_size-overlap_size):
    chunk = text[start:start+chunk_size]
    chunks.append(chunk)

  return chunks

chunks = overlapping_chunks(sample_text)

print("Number of Chunks -> ", len(chunks))
print("First Chunk -> ", chunks[0])

Number of Chunks ->  5
First Chunk ->  

The university library opens at 8 AM and closes at 10 PM.

Students can borrow five books at a tim


In [24]:
import re

def sentence_chunks(text,sentences_per_chunk = 2):
  sentences = re.split(
      r'(?<=[.!?])\s+',
      text.strip()
  )

  chunks = []

  for i in range(0,len(sentences),sentences_per_chunk):
    chunk = " ".join(sentences[i:i+sentences_per_chunk])
    chunks.append(chunk)

  return chunks


chunks = sentence_chunks(sample_text)

for i , chunks in enumerate(chunks):
  print("Chunk ", i+1, " -> ", chunks)
  print()

Chunk  1  ->  The university library opens at 8 AM and closes at 10 PM. Students can borrow five books at a time.

Chunk  2  ->  The borrowing period is fourteen days. Late returns attract a fine of five rupees per day.

Chunk  3  ->  Students must maintain seventy-five percent attendance. The examination registration deadline is November 15.

Chunk  4  ->  Scholarships are available for eligible students.



In [25]:
documents = [
    "The university library closes at 10 PM.",
    "Students must maintain 75 percent attendance.",
    "The examination registration deadline is November 15.",
    "The hostel provides vegetarian and non-vegetarian meals.",
    "Scholarships are available for eligible students.",
    "The cricket team won the inter-college tournament.",
    "The computer science department offers AI courses.",
    "The university provides free Wi-Fi in the library.",
    "Students can borrow five books at a time.",
    "Hostel residents must return before 11 PM."
]


embeddings = embedder.encode(
    documents,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

print("Embeddings Shape -> ", embeddings.shape)

dimnesion = embeddings.shape[1]

index = faiss.IndexFlatIP(dimnesion)

index.add(embeddings)

print("Vectors Stored -> ",index.ntotal)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Shape ->  (10, 384)
Vectors Stored ->  10


In [26]:
query = "When does the library shut ?"

query_embedding = embedder.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

scores , indices = index.search(query_embedding,k=3)

for score , index in zip(scores[0],indices[0]):
  print("Score -> ", score)
  print("Index -> ", index)
  print("Document -> ", documents[index])
  print()

Score ->  0.72659665
Index ->  0
Document ->  The university library closes at 10 PM.

Score ->  0.38131118
Index ->  9
Document ->  Hostel residents must return before 11 PM.

Score ->  0.28529528
Index ->  7
Document ->  The university provides free Wi-Fi in the library.



In [27]:
# Load the Generator

from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM,
)

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(model_name,
                                             dtype = (torch.float16 if torch.backends.mps.is_available() else torch.float32),
                                              trust_remote_code=True)



In [29]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

model.to(device)

model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [30]:
def llm(messages, max_new_tokens=160):
    """
    Run local LLM inference using Hugging Face Transformers.

    Features:
    - No paid API
    - No API key
    - Supports system, user, and assistant messages
    - Disables Qwen3 thinking mode
    - Uses deterministic greedy decoding
    - Explicitly supplies attention mask
    """

    # --------------------------------------------------
    # 1. Convert chat messages into model input
    # --------------------------------------------------

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt"
    )

    # Move input tensors to the model's device
    inputs = {
        key: value.to(next(model.parameters()).device)
        for key, value in inputs.items()
    }

    # --------------------------------------------------
    # 2. Generate response
    # --------------------------------------------------

    with torch.inference_mode():

        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],

            max_new_tokens=max_new_tokens,

            # Deterministic generation
            do_sample=False,

            # Explicitly disable sampling parameters
            temperature=None,
            top_p=None,
            top_k=None,

            pad_token_id=tokenizer.eos_token_id
        )

    # --------------------------------------------------
    # 3. Decode only newly generated tokens
    # --------------------------------------------------

    input_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

reponse = llm([
     {"role" : "system",
      "content" : "You are a helpful assistant"
      },
     {
         "role" : "user",
         "content" : "Explain the process of coding in one sentence"
     }
 ])

print(reponse)


Coding is the process of creating software or programs by writing code in a programming language.


In [31]:
university_documents = [
    {
        "id": "DOC001",
        "text": "The minimum attendance requirement for undergraduate students is 75 percent.",
        "category": "attendance"
    },
    {
        "id": "DOC002",
        "text": "Students with attendance below 75 percent are not eligible for examinations unless an officially approved exemption applies.",
        "category": "attendance"
    },
    {
        "id": "DOC003",
        "text": "The university library opens at 8 AM and closes at 10 PM.",
        "category": "library"
    },
    {
        "id": "DOC004",
        "text": "Students may borrow a maximum of five library books.",
        "category": "library"
    },
    {
        "id": "DOC005",
        "text": "The standard library borrowing period is fourteen days.",
        "category": "library"
    },
    {
        "id": "DOC006",
        "text": "Late return of library books attracts a fine of five rupees per day.",
        "category": "library"
    },
    {
        "id": "DOC007",
        "text": "The examination registration deadline is November 15, 2026.",
        "category": "examination"
    },
    {
        "id": "DOC008",
        "text": "Late examination registration attracts a penalty of five hundred rupees.",
        "category": "examination"
    },
    {
        "id": "DOC009",
        "text": "The hostel curfew is 11 PM.",
        "category": "hostel"
    },
    {
        "id": "DOC010",
        "text": "Scholarship applications must be submitted before December 10, 2026.",
        "category": "scholarship"
    },
    {
        "id": "DOC011",
        "text": "The university provides merit scholarships for academically outstanding students.",
        "category": "scholarship"
    },
    {
        "id": "DOC012",
        "text": "The computer science department offers courses in artificial intelligence and machine learning.",
        "category": "academics"
    }
]

In [32]:
texts = [
    doc["text"]
    for doc in university_documents
]

document_vectors = embedder.encode(
    texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

dimension = document_vectors.shape[1]

rag_index = faiss.IndexFlatIP(dimension)

rag_index.add(document_vectors)

print("Vectors Stored -> ", rag_index.ntotal)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Vectors Stored ->  12


In [33]:
def retrive(query,top_k = 3):
  query_embedding = embedder.encode(
      [query],
      normalize_embeddings=True,
      convert_to_numpy=True
  ).astype("float32")

  scores , indices = rag_index.search(
      query_embedding,
      k=top_k
  )

  results = []

  for score , index in zip(scores[0], indices[0]):
    doc = university_documents[int(index)]
    results.append(
        {
            "id" : doc["id"],
            "text" : doc["text"],
            "category" : doc["category"],
            "score" : float(score)
        }
    )

  return results


results = retrive("What is the deadline for registration?")

for result in results:
  print(
      "Id -> ", result["id"],
      "\nText -> ", result["text"],
      "\nCategory -> ", result["category"],
      "\nScore -> ", result["score"],
      "\n"
  )

Id ->  DOC007 
Text ->  The examination registration deadline is November 15, 2026. 
Category ->  examination 
Score ->  0.698294997215271 

Id ->  DOC010 
Text ->  Scholarship applications must be submitted before December 10, 2026. 
Category ->  scholarship 
Score ->  0.4029565751552582 

Id ->  DOC005 
Text ->  The standard library borrowing period is fourteen days. 
Category ->  library 
Score ->  0.31251639127731323 

